# League market selection · 2025/26 season-ahead evaluation

Run with Python, NumPy, pandas, SciPy and XGBoost. Set each league's CSV filenames in the configuration cell. Place files beside this notebook or in upload/. The pilot development windows stay fixed. The 2025/26 season begins a separate evaluation block. The policy can use earlier completed 2025/26 matches for lagged form, while model fitting, calibration, market qualification and threshold selection use earlier seasons.

Treat the Premier League 2025/26 analysis as retrospective because its earlier holdout results informed the change from 2.5 to 1.5 goals. Report LaLiga separately and call its 2025/26 results unseen only if they did not inform any experiment choices. A pooled comparison including the Premier League is retrospective.


In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar
from scipy.special import softmax
from scipy.stats import poisson
from xgboost import XGBClassifier

LEAGUES = {
    "Premier League": {
        "2023/24": "E 2023 to 2024.csv",
        "2024/25": "E 2024 to 2025.csv",
        "2025/26": "E 2025 to 2026.csv",
    },
    "LaLiga": {
        "2023/24": "SP1_2324.csv",
        "2024/25": "SP1_2425.csv",
        "2025/26": "SP1_2526.csv",
    },
}
TRAIN_END, CAL_END, RANK_MID, RANK_END, THRESHOLD_END, TEST_START, TEST_END = map(pd.Timestamp, [
    "2024-08-01", "2024-12-01", "2025-01-15", "2025-02-15", "2025-04-01", "2025-08-01", "2026-06-01",
])
MIN_WINDOW = 25
MIN_PICKS = 15
MIN_COVERAGE = .20
HIT_TARGET = .60
THRESHOLDS = np.round(np.arange(.50, .901, .025), 3)
MARKETS = {"1X2": ["H", "D", "A"], "O/U 1.5": [0, 1], "BTTS": [0, 1]}
FORM = [f"{side}_{stat}_last_3" for side in ("home", "away")
        for stat in ("points", "goals_scored", "goals_conceded")]
from collections import defaultdict
import csv
from datetime import datetime
from pathlib import Path

ODDS_COLUMNS = []
def load_fixtures(season_files):
    fixtures = []
    seen = set()
    for season, filename in season_files.items():
        path = Path(filename)
        if not path.is_file():
            path = Path("upload") / filename
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for line, row in enumerate(csv.DictReader(stream), start=2):
                try:
                    kickoff = datetime.strptime(
                        f"{row['Date']} {row['Time']}", "%d/%m/%Y %H:%M"
                    )
                    home, away = row["HomeTeam"].strip(), row["AwayTeam"].strip()
                    hg, ag = int(row["FTHG"]), int(row["FTAG"])
                    if not home or not away or home == away or min(hg, ag) < 0:
                        raise ValueError("Invalid teams or score")
                    if row["FTR"] != ("H" if hg > ag else "A" if hg < ag else "D"):
                        raise ValueError("Result disagrees with full-time score")
                except (KeyError, ValueError, TypeError, AttributeError) as error:
                    raise ValueError(f"{path}:{line}: {error}") from error
                fixture_id = f"{season}|{home}|{away}"
                if fixture_id in seen:
                    raise ValueError(f"Duplicate fixture: {fixture_id}")
                seen.add(fixture_id)
                fixtures.append(dict(
                    fixture_id=fixture_id, season=season, kickoff=kickoff,
                    home_team=home, away_team=away, home_goals=hg, away_goals=ag,
                    **{column: row.get(column, "") for column in ODDS_COLUMNS}
                ))
    if not fixtures:
        raise ValueError("No fixtures found")
    return sorted(fixtures, key=lambda row: (row["kickoff"], row["fixture_id"]))

def build_form_features(completed, targets, window=3):
    if not isinstance(window, int) or window < 1:
        raise ValueError("The form window must be a positive integer")

    history = defaultdict(list)
    for match in sorted(completed, key=lambda row: row["kickoff"]):
        hg, ag = match["home_goals"], match["away_goals"]
        for side, scored, conceded in (("home", hg, ag), ("away", ag, hg)):
            points = 3 if scored > conceded else 1 if scored == conceded else 0
            history[match[f"{side}_team"]].append(
                (match["kickoff"].date(), points, scored, conceded)
            )

    rows = []
    for match in sorted(targets, key=lambda row: (row["kickoff"], row["fixture_id"])):
        features = {key: match[key] for key in (
            "fixture_id", "season", "kickoff", "home_team", "away_team"
        )}
        for side in ("home", "away"):
            prior = [
                entry for entry in history[match[f"{side}_team"]]
                if entry[0] < match["kickoff"].date()
            ][-window:]
            features[f"{side}_prior_count_capped"] = len(prior)
            for index, name in enumerate(("points", "goals_scored", "goals_conceded"), 1):
                features[f"{side}_{name}_last_{window}"] = (
                    sum(entry[index] for entry in prior) if len(prior) == window else None
                )
        features["form_ready"] = all(
            features[f"{side}_prior_count_capped"] == window for side in ("home", "away")
        )
        rows.append(features)
    return rows


## Forecasting models

Fit historical class frequencies, Dixon–Coles, and one XGBoost classifier per market and league. Dixon–Coles sums scoreline mass for each outcome and uses neutral effects for unseen teams. Compare raw and temperature-scaled probabilities. Temperature is fitted only on the calibration block. Binary outcome 1 means Over 1.5 or BTTS Yes. Score forecasts for every fixture in the evaluation pool with complete form.


In [ ]:
def fit_dixon_coles(train):
    teams = sorted(set(train.home_team) | set(train.away_team))
    lookup = {team: i for i, team in enumerate(teams)}
    n = len(teams)

    def team_ids(series):
        return np.array([lookup.get(team, n) for team in series], dtype=int)

    def unpack(parameters):
        attack = parameters[:n]
        defence = parameters[n:2*n]
        attack = np.r_[attack - attack.mean(), 0.]
        defence = np.r_[defence - defence.mean(), 0.]
        intercept, advantage, dependence = parameters[-3:]
        max_base = np.exp(intercept + attack.max() + defence.max())
        max_home = max_base * np.exp(advantage)
        lower = -0.99 / max(max_home, max_base)
        upper = 0.99 * min(1., 1. / (max_home * max_base))
        rho = dependence * (upper if dependence >= 0 else -lower)
        return attack, defence, intercept, advantage, rho

    def rates(parameters, home, away):
        attack, defence, intercept, advantage, rho = unpack(parameters)
        return (np.exp(intercept + advantage + attack[home] + defence[away]),
                np.exp(intercept + attack[away] + defence[home]), rho)

    def correction(h, a, home_rate, away_rate, rho):
        return np.select(
            [(h == 0) & (a == 0), (h == 0) & (a == 1),
             (h == 1) & (a == 0), (h == 1) & (a == 1)],
            [1-home_rate*away_rate*rho, 1+home_rate*rho,
             1+away_rate*rho, 1-rho], default=1.)

    home, away = team_ids(train.home_team), team_ids(train.away_team)
    h, a = train.home_goals.to_numpy(), train.away_goals.to_numpy()

    def loss(parameters):
        hr, ar, rho = rates(parameters, home, away)
        log_probability = (poisson.logpmf(h, hr) + poisson.logpmf(a, ar)
                           + np.log(correction(h, a, hr, ar, rho)))
        return -log_probability.sum()

    # Fix one free coordinate per centered effect and constrain the dependence coordinate.
    bounds = ([(-2., 2.)] * (n-1) + [(0., 0.)]) * 2 + [(-2., 2.), (-2., 2.), (-1., 1.)]
    result = minimize(loss, np.zeros(2*n+3), bounds=bounds, method="L-BFGS-B")
    if not result.success:
        raise RuntimeError(result.message)

    def score_probabilities(frame):
        hr, ar, rho = rates(result.x, team_ids(frame.home_team), team_ids(frame.away_team))
        max_goals = max(10, int(poisson.ppf(1-1e-10, max(hr.max(), ar.max()))))
        h_grid, a_grid = np.ogrid[:max_goals+1, :max_goals+1]
        hr, ar = hr[:, None, None], ar[:, None, None]
        grid = (poisson.pmf(h_grid, hr) * poisson.pmf(a_grid, ar)
                * correction(h_grid, a_grid, hr, ar, rho))
        mass = grid.sum(axis=(1, 2), keepdims=True)
        assert np.allclose(mass, 1, atol=1e-8, rtol=0)
        grid /= mass
        groups = {
            "1X2": [h_grid > a_grid, h_grid == a_grid, h_grid < a_grid],
            "O/U 1.5": [h_grid+a_grid < 2, h_grid+a_grid >= 2],
            "BTTS": [(h_grid == 0) | (a_grid == 0), (h_grid > 0) & (a_grid > 0)],
        }
        return {name: np.column_stack([grid[:, mask].sum(axis=1) for mask in masks])
                for name, masks in groups.items()}
    return score_probabilities

In [ ]:
def loss(p, y):
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1)).mean()

def scaled(p, log_t):
    return softmax(np.log(np.clip(p, 1e-15, 1)) / np.exp(log_t), axis=1)

def league_forecasts(league, season_files):
    completed = load_fixtures(season_files)
    if sum(m["season"] == "2025/26" and TEST_START <= m["kickoff"] < TEST_END
           for m in completed) != 380:
        raise ValueError(f"{league}: supply all 380 fixtures for 2025/26")
    rows = pd.DataFrame(build_form_features(completed, completed))
    scores = pd.DataFrame(completed)[["fixture_id", "home_goals", "away_goals"]]
    rows = rows.merge(scores, validate="one_to_one", on="fixture_id")
    h, a = rows.home_goals, rows.away_goals
    rows["1X2"] = np.select([h > a, h < a], ["H", "A"], default="D")
    rows["O/U 1.5"] = (h+a >= 2).astype(int)
    rows["BTTS"] = ((h > 0) & (a > 0)).astype(int)
    train = rows.loc[rows.kickoff.lt(TRAIN_END)]
    fit = train.loc[train.form_ready]
    future = rows.loc[rows.form_ready & rows.kickoff.ge(TRAIN_END)
                      & rows.kickoff.lt(TEST_END)].reset_index(drop=True)
    d = future.kickoff
    masks = {
        "cal": d.lt(CAL_END),
        "rank1": d.ge(CAL_END) & d.lt(RANK_MID),
        "rank2": d.ge(RANK_MID) & d.lt(RANK_END),
        "threshold": d.ge(RANK_END) & d.lt(THRESHOLD_END),
        "test": d.ge(TEST_START),
    }
    counts = {name: int(mask.sum()) for name, mask in masks.items()}
    if len(fit) < MIN_WINDOW or min(counts.values()) < MIN_WINDOW:
        raise ValueError(f"{league}: insufficient fixture support. Training={len(fit)}, windows={counts}")
    print(league, "training", len(fit), counts)
    dc = fit_dixon_coles(train)(future)
    forecasts = {}
    for market, labels in MARKETS.items():
        code = {label: i for i, label in enumerate(labels)}
        train_y = fit[market].map(code).to_numpy(dtype=int)
        y = future[market].map(code).to_numpy(dtype=int)
        freq = np.bincount(train_y, minlength=len(labels))
        if not freq.all():
            raise ValueError(f"{league}/{market}: training class missing")
        reference = np.tile(freq/freq.sum(), (len(future), 1))
        model = XGBClassifier(n_estimators=100, max_depth=2, learning_rate=.05,
                              random_state=42, n_jobs=1,
                              eval_metric="mlogloss" if len(labels)==3 else "logloss")
        model.fit(fit[FORM], train_y)
        np.testing.assert_array_equal(model.classes_, np.arange(len(labels)))
        options = {}
        for name, p in {"Dixon–Coles": dc[market],
                        "XGBoost": model.predict_proba(future[FORM])}.items():
            np.testing.assert_allclose(p.sum(axis=1), 1, atol=1e-6)
            cal = masks["cal"]
            result = minimize_scalar(lambda t: loss(scaled(p[cal], t), y[cal]),
                                     bounds=(np.log(.25), np.log(4)), method="bounded")
            if not result.success:
                raise RuntimeError(result.message)
            options[(name, "Raw")] = (p, 1.)
            options[(name, "Scaled")] = (scaled(p, result.x), np.exp(result.x))
        forecasts[market] = (labels, y, reference, options)
    return future, masks, forecasts

assert TRAIN_END < CAL_END < RANK_MID < RANK_END < THRESHOLD_END < TEST_START < TEST_END
runs = {league: league_forecasts(league, files) for league, files in LEAGUES.items()}


## Choose markets and freeze both policies

Skill is 1 − model log loss / frequency baseline log loss. Within each league and market, choose the model variant by mean skill over the two ranking windows. A league qualifies a market only if the winning variant has positive skill in both windows. The common policy uses the same per-league forecasts but qualifies a market only when its equal-league mean skill is positive in both windows.

Tune a threshold separately per policy on the same pooled threshold-validation fixtures. Take at most one prediction per fixture, using the highest eligible probability and alphabetical market order to break ties. Choose the widest supported coverage at a validation hit rate of at least 60%. If no threshold meets the hit-rate target, choose the highest validation hit rate among supported thresholds. If no threshold is supported, abstain. Report evaluation hit rate and coverage together. Include a shared-cutoff sensitivity comparison at 0.625 to distinguish market-list changes from threshold changes.


In [ ]:
rank_rows, candidates, probability_rows, pools, forecast_rows = [], [], [], [], []
for league, (future, masks, markets) in runs.items():
    pool = future[["fixture_id", "kickoff"]].assign(league=league)
    pool["period"] = np.select([masks["threshold"], masks["test"]],
                               ["validation", "evaluation"], default="earlier")
    pools.append(pool)
    for market, (labels, y, reference, options) in markets.items():
        windows = [masks["rank1"], masks["rank2"]]
        baseline = [loss(reference[w], y[w]) for w in windows]
        def skill(p):
            return np.array([1-loss(p[w], y[w])/b for w, b in zip(windows, baseline)])
        model, variant = max(options, key=lambda key: skill(options[key][0]).mean())
        p, temperature = options[(model, variant)]
        s1, s2 = skill(p)
        rank_rows.append(dict(league=league, market=market, model=model, variant=variant,
                              temperature=temperature, skill_window_1=s1, skill_window_2=s2,
                              league_eligible=min(s1, s2)>0))
        best = p.argmax(axis=1)
        candidates.append(pool.assign(
            market=market, outcome=np.array(labels)[best], confidence=p.max(axis=1),
            correct=best == y, baseline_correct=reference.argmax(axis=1) == y,
        ))
        test = masks["test"]
        for index, label in enumerate(labels):
            forecast_rows.append(pool.loc[test, ["league", "fixture_id"]].assign(
                market=market, outcome=str(label), probability=p[test, index],
                reference=reference[test, index], observed=(y[test] == index).astype(int),
            ))
        one_hot = np.eye(len(labels))[y[test]]
        for name, q in {"Chosen forecast": p, "Historical frequency": reference}.items():
            probability_rows.append(dict(
                league=league, market=market, forecast=name, fixtures=int(test.sum()),
                log_loss=loss(q[test], y[test]),
                brier=np.square(q[test]-one_hot).sum(axis=1).mean(),
            ))

ranks = pd.DataFrame(rank_rows)
skill_cols = ["skill_window_1", "skill_window_2"]
common = ranks.groupby("market")[skill_cols].mean().gt(0).all(axis=1)
ranks["common_eligible"] = ranks.market.map(common)
display(ranks)
pool = pd.concat(pools, ignore_index=True)
candidates = pd.concat(candidates, ignore_index=True).merge(
    ranks[["league", "market", "league_eligible", "common_eligible"]],
    on=["league", "market"], validate="many_to_one",
)


In [ ]:
def summarize(picks, eligible):
    n = len(picks)
    return dict(fixtures=eligible, picks=n, correct=int(picks.correct.sum()),
                coverage=n/eligible if eligible else np.nan,
                hit_rate=picks.correct.mean() if n else np.nan,
                matched_baseline=picks.baseline_correct.mean() if n else np.nan)

grid_rows, evaluation_rows, exports = [], [], []
for policy, flag in [("League-specific", "league_eligible"), ("Common", "common_eligible")]:
    best = (candidates.loc[candidates[flag]]
            .sort_values(["confidence", "market"], ascending=[False, True])
            .drop_duplicates(["league", "fixture_id"]))
    val = best.loc[best.period.eq("validation")]
    test = best.loc[best.period.eq("evaluation")]
    n_val = pool.period.eq("validation").sum()
    grid = pd.DataFrame([
        dict(policy=policy, threshold=t, **summarize(val.loc[val.confidence.ge(t)], n_val))
        for t in THRESHOLDS
    ])
    grid_rows.append(grid)
    supported = grid.loc[grid.picks.ge(MIN_PICKS) & grid.coverage.ge(MIN_COVERAGE)]
    meeting = supported.loc[supported.hit_rate.ge(HIT_TARGET)]
    if len(meeting):
        choice = meeting.sort_values(["coverage", "hit_rate", "threshold"],
                                     ascending=[False, False, True]).iloc[0]
        reason = "Validation target met"
    elif len(supported):
        choice = supported.sort_values(["hit_rate", "coverage", "threshold"],
                                       ascending=[False, False, True]).iloc[0]
        reason = "Supported fallback"
    else:
        choice, reason = None, "No supported threshold"
    threshold = float(choice.threshold) if choice is not None else np.nan
    for mode, t in [("Validation-selected", threshold), ("Shared 0.625", .625)]:
        picks = test.loc[test.confidence.ge(t)].assign(policy=policy, mode=mode)
        exports.append(picks)
        for league in ["All leagues", *LEAGUES]:
            sample = pool.loc[pool.period.eq("evaluation")]
            subset = picks
            if league != "All leagues":
                sample = sample.loc[sample.league.eq(league)]
                subset = picks.loc[picks.league.eq(league)]
            evaluation_rows.append(dict(policy=policy, mode=mode, league=league, threshold=t,
                                        rule=reason if mode=="Validation-selected" else "Fixed comparison",
                                        **summarize(subset, len(sample))))

summary = pd.DataFrame(evaluation_rows)
picks = pd.concat(exports, ignore_index=True)
market_summary = (picks.groupby(["policy", "mode", "league", "market"])
                  .agg(picks=("correct", "size"), correct=("correct", "sum"),
                       hit_rate=("correct", "mean"), matched_baseline=("baseline_correct", "mean"))
                  .reset_index())
display(summary)
display(market_summary)
if len(LEAGUES) == 1:
    print("One-league smoke test: the two market policies are equivalent by construction.")
output = Path("league_experiment_outputs")
output.mkdir(exist_ok=True)
for name, frame in {
    "market_rankings": ranks, "validation_thresholds": pd.concat(grid_rows),
    "evaluation_summary": summary, "evaluation_by_market": market_summary,
    "evaluation_picks": picks, "evaluation_probability_scores": pd.DataFrame(probability_rows),
}.items():
    frame.to_csv(output / f"{name}.csv", index=False)


## Match coverage within each league

Compare the two frozen validation-selected policies at the smaller of their realized selection counts within each league. Keep the highest-confidence picks for each policy, breaking ties by kickoff and fixture ID. This is a retrospective, outcome-blind sensitivity check. The original prospective operating points remain in the evaluation summary. If either policy abstains entirely in a league, the equal-coverage comparison there is undefined.


In [ ]:
matched = []
for league in LEAGUES:
    groups = {
        policy: picks.loc[picks.league.eq(league) & picks.policy.eq(policy)
                          & picks["mode"].eq("Validation-selected")]
        for policy in ("League-specific", "Common")
    }
    k = min(map(len, groups.values()))
    eligible = int((pool.league.eq(league) & pool.period.eq("evaluation")).sum())
    for policy, group in groups.items():
        top = group.sort_values(["confidence", "kickoff", "fixture_id"],
                                ascending=[False, True, True]).head(k)
        matched.append(dict(league=league, policy=policy, **summarize(top, eligible)))
matched = pd.DataFrame(matched)
display(matched)
matched.to_csv(output / "evaluation_matched_coverage.csv", index=False)


## Compare probabilities with matched bookmaker odds

The default adapter reads market-average closing 1X2 odds from the same league CSVs. For Over/Under 1.5 or BTTS, optionally supply bookmaker_odds_exact_markets.csv in the working directory, with columns league, fixture_id, market, outcome, decimal_odds. Supply a single consistent bookmaker/source and snapshot definition per league and market. Fixture IDs use season|home_team|away_team, with team names as in the source CSV. Market values are 1X2, O/U 1.5 and BTTS. Outcomes are H/D/A, 0 (Under or No) and 1 (Over or Yes). Supply all outcomes for a market on the same snapshot. The optional file is for O/U 1.5 and BTTS only. Use odds at the exact 1.5 line. Closing odds are a closing-market reference because their information cutoff can differ from the model's.

The comparison uses proportional margin removal and scores the selected forecasting model, historical frequencies, and bookmaker probabilities on identical fixtures within each league and market. An unavailable odds set yields zero matched fixtures and no comparative scores.


In [ ]:
odds_rows = []
closing_columns = {"H": "AvgCH", "D": "AvgCD", "A": "AvgCA"}
for league, files in LEAGUES.items():
    path = Path(files["2025/26"])
    if not path.is_file():
        path = Path("upload") / path
    with path.open(encoding="utf-8-sig", newline="") as stream:
        for row in csv.DictReader(stream):
            fixture_id = f"2025/26|{row['HomeTeam'].strip()}|{row['AwayTeam'].strip()}"
            for outcome, column in closing_columns.items():
                odds_rows.append(dict(league=league, fixture_id=fixture_id, market="1X2",
                                      outcome=outcome, decimal_odds=row.get(column)))
book = pd.DataFrame(odds_rows)
extra_path = Path("bookmaker_odds_exact_markets.csv")
keys = ["league", "fixture_id", "market", "outcome"]
if extra_path.is_file():
    extra = pd.read_csv(extra_path, dtype={"outcome": str})
    if not extra.market.isin(["O/U 1.5", "BTTS"]).all() or not extra.outcome.isin(["0", "1"]).all():
        raise ValueError("Optional odds must specify O/U 1.5 or BTTS, with outcomes 0 and 1")
    book = pd.concat([book, extra[keys + ["decimal_odds"]]], ignore_index=True)
book["decimal_odds"] = pd.to_numeric(book.decimal_odds, errors="coerce")
forecasts = pd.concat(forecast_rows, ignore_index=True)
joined = forecasts.merge(book, on=keys, how="left", validate="one_to_one")
odds_summary = []
for (league, market), group in joined.groupby(["league", "market"]):
    valid = np.isfinite(group.decimal_odds) & group.decimal_odds.gt(1)
    complete = valid.groupby(group.fixture_id).transform("all")
    subset = group.loc[complete].copy()
    n = subset.fixture_id.nunique()
    if not n:
        odds_summary.append(dict(league=league, market=market, forecast="Odds unavailable",
                                 fixtures=0, log_loss=np.nan, brier=np.nan))
        continue
    inverse = 1 / subset.decimal_odds
    subset["bookmaker"] = inverse / inverse.groupby(subset.fixture_id).transform("sum")
    observed = subset.observed.to_numpy()
    for name in ("probability", "reference", "bookmaker"):
        prob = subset[name].to_numpy()
        odds_summary.append(dict(
            league=league, market=market, forecast=name, fixtures=n,
            log_loss=-np.log(np.clip(prob[observed == 1], 1e-15, 1)).mean(),
            brier=np.square(prob-observed).sum()/n,
        ))
odds_summary = pd.DataFrame(odds_summary)
display(odds_summary)
odds_summary.to_csv(output / "evaluation_matched_bookmaker.csv", index=False)
